# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

## Business Scenario

You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.


Your task is to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


## Dataset Description: Hotel Bookings

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance

### Data Dictionary

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Load the `hotels.csv` file
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In your markdown:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

url = 'https://raw.githubusercontent.com/vandanara/UofUtah_IS4487/refs/heads/main/DataSets/hotels.csv'
df = pd.read_csv(url)

# Optional column drops
df = df.drop(columns=['reservation_status', 'reservation_status_date', 'assigned_room_type'])

# Impute missing values
df['children'] = df['children'].fillna(0)
df['babies'] = df['babies'].fillna(0)

df['country'] = df['country'].fillna('Unknown')
df['agent'] = df['agent'].fillna('Unknown').astype(str)
df['company'] = df['company'].fillna('Unknown').astype(str)
df['meal'] = df['meal'].fillna('Unknown')

# Encode categoricals
categorical_cols = df.select_dtypes(include='object').columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Features + target
X = df_encoded.drop('is_canceled', axis=1)
y = df_encoded['is_canceled']

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Original data:")
print(df.head())

print("\nEncoded data:")
print(df_encoded.head())

print("\nShapes after train/test split:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print(df.shape)


Original data:
          hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  \
0                        0                     0       2  ...   
1                  

### ✍️ Your Response:
1. The original hotels.csv dataset contains 119,390 rows and 29 columns.

2. Numerical features: integers and floats such as lead_time, adults, children, babies, stays_in_weekend_nights, stays_in_week_nights, adr. Categorical features: hotel type, meal plan, market segment, country, distribution channel, customer type, deposit type, agent, company.

3. The data preparation process involved removing unneeded variables, filling in missing information, and transforming categorical data for modeling. Missing numeric values, such as children and babies, were imputed with zeros, while missing categorical values like country, agent, company, and meal were replaced with the label “Unknown” to maintain consistency. Several columns not needed for modeling were removed, and categorical fields were converted into usable numerical form through one-hot encoding. After cleaning, the dataset was split into training and testing sets to support model development and evaluation.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In your markdown:
1. How accurate is this model?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train the Naïve Bayes model
nb = BernoulliNB()
nb.fit(X_train, y_train)

# Predict on the test set
y_pred = nb.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.7908255856157691

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.90      0.84     22550
           1       0.78      0.60      0.68     13267

    accuracy                           0.79     35817
   macro avg       0.79      0.75      0.76     35817
weighted avg       0.79      0.79      0.78     35817


Confusion Matrix:
 [[20367  2183]
 [ 5309  7958]]


### ✍️ Your Response:
1. The Naïve Bayes model delivers ~79% accuracy

2. Even with modest performance, this model still drives operational value. It can act as an early-warning tool, supporting frontline teams with real-time decision support. For instance, it can flag reservations with a higher likelihood of no-show so the hotel can tighten overbooking strategies, allocate rooms more efficiently, and adjust staffing levels before demand hits the floor. It also provides a quick-response mechanism for revenue managers to prioritize follow-ups, reminders, or re-confirmation calls to guests who are more likely to cancel or not arrive

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use RBF kernel)
- Make predictions and evaluate with classification metrics

### In your markdown:
1. How well does the model perform?
2. In what business situations could SVM provide better insights than simpler models?


In [10]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import numpy as np

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sample_size = 8000
indices = np.random.choice(len(X_train_scaled), size=sample_size, replace=False)

X_small = X_train_scaled[indices]
y_small = y_train.iloc[indices]

svm_rbf = SVC(kernel='rbf')
svm_rbf.fit(X_small, y_small)

y_pred = svm_rbf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Accuracy: 0.8068096155456906

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.92      0.86     14907
           1       0.83      0.61      0.70      8971

    accuracy                           0.81     23878
   macro avg       0.81      0.77      0.78     23878
weighted avg       0.81      0.81      0.80     23878


Confusion Matrix:
 [[13781  1126]
 [ 3487  5484]]


### ✍️ Your Response:
1. The RBF SVM is performing better than your Naïve Bayes baseline. Overall accuracy is about 80.4%, up from ~79%. More importantly, it delivers strong precision on both classes (0.80 for shows, 0.82 for no-shows) and improves the overall balance of errors. Class 0 (shows) has 0.92 recall, so it correctly identifies most guests who actually arrive. Class 1 (no-shows) has a 0.60 recall with a 0.70 F1, which is slightly better structured than the Naïve Bayes model for capturing no-shows while keeping false alarms under control. Net-net: this is a step up in performance and robustness versus the simpler model, especially in a setting where patterns are not purely linear.

2. SVM is most useful for the hotel in situations where guest behavior depends on more complex patterns that simpler models can’t capture. For example, it can better identify no-show risks when the probability depends on nonlinear combinations of booking channel, lead time, price, and stay details. It also supports more accurate overbooking decisions by giving the hotel a sharper read on which guests are unlikely to arrive. In addition, SVM can improve targeted outreach efforts by flagging specific high-risk reservations for reminders or follow-ups instead of contacting every guest. Overall, it adds value in any operational or revenue-management scenario where decisions depend on subtle, hard-to-detect behaviors in the data.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a neural network using `MLPClassifier`
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

### In your markdown:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [11]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=50,
    random_state=42)

mlp.fit(X_train_scaled, y_train)
y_pred = mlp.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Accuracy: 0.8691263924951839

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.90      0.90     14907
           1       0.83      0.82      0.82      8971

    accuracy                           0.87     23878
   macro avg       0.86      0.86      0.86     23878
weighted avg       0.87      0.87      0.87     23878


Confusion Matrix:
 [[13440  1467]
 [ 1658  7313]]


### ✍️ Your Response:
1. The neural network outperforms the previous models across the board. It delivered an accuracy of about 86.8%, which is meaningfully higher than both the Naïve Bayes model (~79%) and the RBF SVM (~80%)

2. From an operational standpoint, the business may be hesitant to rely fully on a black-box model. Neural networks are harder to interpret, which can make leadership uneasy when decisions affect staffing, overbooking strategy, or guest communication. However, given the model’s strong performance, it could still be a valuable decision-support tool, especially if used alongside simpler, more interpretable models. The hotel might adopt it in analytics workflow, but still prefers a more transparent model when accountability, explanation, or regulatory confidence is required.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In your markdown:
1. Which model would you recommend for deployment, and why?
2. Consider accuracy, training time, interpretability, and ease of use.


In [12]:
nb_accuracy = 0.7908
svm_accuracy = 0.8045
nn_accuracy = 0.8679
print("Model Accuracy Comparison:")
print("---------------------------")
print(f"Naïve Bayes Accuracy:      {nb_accuracy:.4f}")
print(f"SVM (RBF) Accuracy:         {svm_accuracy:.4f}")
print(f"Neural Network Accuracy:    {nn_accuracy:.4f}")


Model Accuracy Comparison:
---------------------------
Naïve Bayes Accuracy:      0.7908
SVM (RBF) Accuracy:         0.8045
Neural Network Accuracy:    0.8679


### ✍️ Your Response:
1. The neural network is the clear top performer, delivering an accuracy of roughly 86.8%, which outpaces both the Naïve Bayes model (79%) and the SVM (80%). Beyond accuracy, it also provides stronger, more balanced precision and recall across both guest categories, meaning it’s better at correctly identifying both shows and no-shows. That translates directly into tighter operational forecasting, fewer missed revenue opportunities, and more reliable decision-support for the hotel’s core processes. In terms of ROI, the neural network gives the hotel the most lift. It detects more subtle, nonlinear behavior patterns in booking activity, which the simpler models fail to pick up. Even though it’s a black-box model, its superior performance justifies its use in any environment where accuracy drives operational value, like overbooking decisions, staffing plans, and targeted guest outreach.

2. The neural network clearly performed the best. It delivered the highest accuracy at about 86.8%, well above the Naïve Bayes model (79%) and the SVM (80%). It also showed stronger, more balanced precision and recall for both guest show and no-show classes. Overall, it captured the most complex patterns in the data and produced the most reliable predictions, making it the top-performing model in this comparison.

## 6. Final Business Recommendation

### In your markdown:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response:
1. Our analysis shows that the neural network model delivers the strongest predictive performance, achieving significantly higher accuracy than Naïve Bayes or SVM. I recommend implementing this model to improve no-show forecasting, strengthen the overbooking strategy, and optimize staffing and room allocation. While the model operates as a black box, its performance gains justify its use for operational decision-making. To further enhance results, the hotel should collect additional variables such as guest history, booking channel behavior, seasonality, and pricing data. This will help refine predictions and support more proactive revenue and service planning.

2. This project directly supports the learning outcomes I created because it gave me real experience using analytics to evaluate uncertainty and risk, similar to how I would analyze market volatility or portfolio exposure. By comparing Naïve Bayes, SVM, and neural networks, I practiced determining which model best handles unpredictable behavior, which connects to the analytical skills I want to develop for finance and risk analysis. Writing the final recommendation to hotel management also helped me translate complex results into clear, actionable insights. That directly aligns with my goal of communicating data-driven recommendations in consulting and corporate decision-making roles.

In [13]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"

[NbConvertApp] Converting notebook assignment_12_bayes_svm_neural.ipynb to html
[NbConvertApp] Writing 323954 bytes to assignment_12_bayes_svm_neural.html
